<a href="https://colab.research.google.com/github/jaehyeon-gg/deep_learning/blob/main/Weights_Biases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weights & Biases (W&B)

ML 실험 추적, 시각화, 하이퍼파라미터 자동 탐색을 위한 도구입니다.
간단한 코드로
대화형 공유 대시보드를 생성할 수 있습니다. [살펴보기](https://wandb.ai/wandb/wandb_example)

![](https://i.imgur.com/Pell4Oo.png)

## 학습 목표

- W&B의 기본 개념 (Run, Project, Config, Metric)
- PyTorch 모델 학습 과정에서 metric과 예측 결과를 W&B에 기록하는 방법
- Hyperparameter Sweep을 이용한 자동 탐색

## 준비물

- [wandb.ai](https://wandb.ai) 계정
- API key ([wandb.ai/authorize](https://wandb.ai/authorize) 혹은 quickstart 페이지에서 발급)

전체 문서: [Weights & Biases Docs](https://docs.wandb.ai/quickstart)

https://wandb.ai/jaehyeon6874-2

## 1. W&B를 사용하는 이유


W&B는 다음 기능을 제공합니다.

1. **실험 추적**: hyperparameter, metric, 시스템 정보 자동 기록
2. **시각화 대시보드**: 실시간 차트, 비교 view, 인터랙티브 테이블
3. **Sweeps**: hyperparameter 자동 탐색 (grid, random, Bayesian)
4. **협업**: URL 기반 공유, 리포트 작성
5. **Artifacts**: 데이터셋과 모델 버전 관리

본 실습에서는 1, 2, 3을 다룹니다.

## 2. 설치 및 로그인

라이브러리 설치 후 W&B 계정으로 로그인합니다. 첫 실행 시 API key 입력이 요구됩니다.

In [ ]:
!pip install wandb -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 79.5 MB/s eta 0:00:00


In [ ]:
import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jaehyeon6874 (jaehyeon6874-2) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Quickstart

W&B 사용은 세 단계로 구성됩니다.

1. `wandb.init()`: 새로운 run을 시작하고 hyperparameter를 `config`로 등록
2. `wandb.log()`: 학습 또는 평가 중 metric 기록
3. `wandb.finish()`: run 종료

아래 예제는 실제 모델 학습 대신 무작위 값으로 metric을 생성합니다.
각 함수의 기능을 확인해봅시다.

In [ ]:
import random

# 5개의 시뮬레이션 실험 실행
total_runs = 5
for run in range(total_runs):
    # 1. Run 시작: project와 run 이름, hyperparameter 등록
    wandb.init(
        project="basic-intro",         # project 이름 (없으면 자동 생성)
        name=f"experiment_{run}",      # run 이름 (생략 시 무작위 이름 부여)
        config={
            "learning_rate": 0.02,
            "architecture": "CNN",
            "dataset": "CIFAR-100",
            "epochs": 10,
        }
    )

    # 학습 루프 시뮬레이션: epoch마다 acc/loss를 무작위로 생성
    epochs = 10
    offset = random.random() / 5
    for epoch in range(2, epochs):
        acc = 1 - 2 ** -epoch - random.random() / epoch - offset
        loss = 2 ** -epoch + random.random() / epoch + offset

        # 2. Metric 기록: dict 형태로 자유롭게 key-value 전달
        wandb.log({"acc": acc, "loss": loss})

    # 3. Run 종료 (Jupyter 환경에서 다음 run으로 정상 전환을 위해 필요)
    wandb.finish()

acc,▁▂▆███▇█
loss,██▃▁▂▁▁▁
acc,0.78351
loss,0.25362


acc,▁▄▆▇▇▇██
loss,█▅▄▃▂▁▁▂
acc,0.81032
loss,0.20667


acc,▂▁▆▇▆▇█▇
loss,█▆▂▃▁▁▂▁
acc,0.78697
loss,0.16872


acc,▁▃▆▇████
loss,█▆▄▁▂▂▂▁
acc,0.9313
loss,0.11982


acc,▁▂▇▇▆▇██
loss,█▇▅▃▃▁▁▂
acc,0.7451
loss,0.28001


위의 👆 wandb 링크 중 하나를 클릭하면 인터랙티브 대시보드로 연결됩니다.

### 대시보드 확인 사항

- 5개 experiment run이 자동으로 색상 구분되어 표시됩니다.
- 차트 위에 커서를 올리면 정확한 값이 툴팁으로 나타납니다.
- 좌측 사이드바에서 run을 선택적으로 표시/숨김할 수 있습니다.
- 우측 상단의 Add panels 버튼으로 차트를 추가할 수 있습니다.

## 4. PyTorch 모델 학습

MNIST 데이터셋으로 간단한 MLP 분류기를 학습하면서 W&B의 주요 기능을 사용합니다.

- Train/Validation metric을 실시간으로 W&B에 전송
- 예측 결과 이미지를 `wandb.Table`로 기록
- Dropout 값을 변경하여 5회의 실험 비교

Run 실행 시 다음 정보가 자동으로 기록됩니다.

- Metric
- 시스템 정보 (GPU/CPU 사용량)
- Hyperparameter
- 터미널 출력
- 인터랙티브 테이블

### 4.1 Dataloader와 모델 정의

In [ ]:
import wandb
import math
import random
import torch, torchvision
import torch.nn as nn
import torchvision.transforms as T

device = "cuda:0" if torch.cuda.is_available() else "cpu"

def get_dataloader(is_train, batch_size, slice=5):
    """MNIST dataloader (slice=5: 데이터의 1/5만 사용하여 학습 시간 단축)"""
    full_dataset = torchvision.datasets.MNIST(root=".", train=is_train, transform=T.ToTensor(), download=True)
    sub_dataset = torch.utils.data.Subset(full_dataset, indices=range(0, len(full_dataset), slice))
    loader = torch.utils.data.DataLoader(
        dataset=sub_dataset,
        batch_size=batch_size,
        shuffle=True if is_train else False,
        pin_memory=True, num_workers=2
    )
    return loader

def get_model(dropout):
    """간단한 MLP 모델"""
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(256, 10),
    ).to(device)
    return model

def validate_model(model, valid_dl, loss_func, log_images=False, batch_idx=0):
    """Validation 수행. log_images=True인 경우 한 batch의 예측을 wandb.Table로 기록"""
    model.eval()
    val_loss = 0.
    with torch.inference_mode():
        correct = 0
        for i, (images, labels) in enumerate(valid_dl):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += loss_func(outputs, labels) * labels.size(0)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()

            # 마지막 epoch의 한 batch만 이미지 테이블로 기록
            if i == batch_idx and log_images:
                log_image_table(images, predicted, labels, outputs.softmax(dim=1))
    return val_loss / len(valid_dl.dataset), correct / len(valid_dl.dataset)

def log_image_table(images, predicted, labels, probs):
    """wandb.Table을 생성하여 (이미지, 예측, 정답, 클래스별 점수)를 기록"""
    table = wandb.Table(columns=["image", "pred", "target"] + [f"score_{i}" for i in range(10)])
    for img, pred, targ, prob in zip(images.to("cpu"), predicted.to("cpu"), labels.to("cpu"), probs.to("cpu")):
        table.add_data(wandb.Image(img[0].numpy() * 255), pred, targ, *prob.numpy())
    wandb.log({"predictions_table": table}, commit=False)

### 4.2 학습 실행

Dropout 값을 무작위로 변경하면서 5회 학습합니다. 각 run마다 다음 작업이 수행됩니다.

1. `wandb.init()`으로 새 run 시작 (dropout은 매번 무작위 값)
2. `wandb.config`로 hyperparameter 접근
3. 매 step마다 `wandb.log()`로 train loss 기록
4. epoch 종료 시점에 validation loss/accuracy 기록
5. 마지막 epoch에서 예측 이미지를 테이블로 기록
6. `wandb.summary`로 최종 metric (test_accuracy) 기록

In [ ]:
# Dropout 값을 변경하여 5회 실험
for run in range(5):
    # 새 run 시작
    wandb.init(
        project="pytorch-intro",
        name=f"experiment_{run}",
        config={
            "epochs": 10,
            "batch_size": 128,
            "lr": 1e-3,
            "dropout": random.uniform(0.01, 0.80),
        },
    )

    # config 객체를 통한 hyperparameter 접근
    config = wandb.config

    train_dl = get_dataloader(is_train=True, batch_size=config.batch_size)
    valid_dl = get_dataloader(is_train=False, batch_size=2 * config.batch_size)
    n_steps_per_epoch = math.ceil(len(train_dl.dataset) / config.batch_size)

    model = get_model(config.dropout)
    loss_func = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

    # 학습 루프
    example_ct = 0 # 학습 시작부터 지금까지 모델이 본 누적 샘플(데이터) 개수
    step_ct = 0
    for epoch in range(config.epochs):
        model.train()
        for step, (images, labels) in enumerate(train_dl):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            train_loss = loss_func(outputs, labels)
            optimizer.zero_grad()
            train_loss.backward()
            optimizer.step()

            example_ct += len(images)
            # 'train/' prefix를 사용하여 W&B UI에서 자동 그룹화
            metrics = {
                "train/train_loss": train_loss,
                "train/epoch": (step + 1 + (n_steps_per_epoch * epoch)) / n_steps_per_epoch,
                "train/example_ct": example_ct,
            }

            if step + 1 < n_steps_per_epoch:
                wandb.log(metrics)

            step_ct += 1

        # Validation
        val_loss, accuracy = validate_model(model, valid_dl, loss_func, log_images=(epoch == (config.epochs - 1)))
        val_metrics = {"val/val_loss": val_loss, "val/val_accuracy": accuracy}
        wandb.log({**metrics, **val_metrics})

        print(f"Train Loss: {train_loss:.3f}, Valid Loss: {val_loss:3f}, Accuracy: {accuracy:.2f}")

    wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...

train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
train/example_ct,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇████
train/train_loss,█▄▂▁▂▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/val_accuracy,▁▄▄▆▇▇▆▇▇█
val/val_loss,█▅▄▃▂▂▁▁▁▁
train/epoch,10
train/example_ct,120000
train/train_loss,0.05466
val/val_accuracy,0.955
val/val_loss,0.16232


Train Loss: 0.306, Valid Loss: 0.275497, Accuracy: 0.92
Train Loss: 0.125, Valid Loss: 0.216495, Accuracy: 0.94
Train Loss: 0.145, Valid Loss: 0.198276, Accuracy: 0.94
Train Loss: 0.130, Valid Loss: 0.174468, Accuracy: 0.94
Train Loss: 0.189, Valid Loss: 0.167734, Accuracy: 0.94
Train Loss: 0.087, Valid Loss: 0.161889, Accuracy: 0.95
Train Loss: 0.053, Valid Loss: 0.150338, Accuracy: 0.95
Train Loss: 0.038, Valid Loss: 0.152912, Accuracy: 0.95
Train Loss: 0.048, Valid Loss: 0.155276, Accuracy: 0.95
Train Loss: 0.035, Valid Loss: 0.149426, Accuracy: 0.95


train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇███
train/example_ct,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇██
train/train_loss,█▆█▅▇▄▄▃▃▄▄▄▃▃▄▄▅▃▃▃▂▂▂▂▂▃▂▁▂▂▂▂▂▁▂▁▁▁▁▁
val/val_accuracy,▁▅▅▆▆▇▇█▇█
val/val_loss,█▅▄▂▂▂▁▁▁▁
train/epoch,10
train/example_ct,120000
train/train_loss,0.03535
val/val_accuracy,0.953
val/val_loss,0.14943


Train Loss: 0.238, Valid Loss: 0.286439, Accuracy: 0.92
Train Loss: 0.075, Valid Loss: 0.222541, Accuracy: 0.93
Train Loss: 0.151, Valid Loss: 0.199578, Accuracy: 0.95
Train Loss: 0.123, Valid Loss: 0.179040, Accuracy: 0.95
Train Loss: 0.062, Valid Loss: 0.168738, Accuracy: 0.95
Train Loss: 0.057, Valid Loss: 0.157772, Accuracy: 0.95
Train Loss: 0.063, Valid Loss: 0.165740, Accuracy: 0.95
Train Loss: 0.030, Valid Loss: 0.151175, Accuracy: 0.95
Train Loss: 0.080, Valid Loss: 0.154726, Accuracy: 0.95
Train Loss: 0.029, Valid Loss: 0.148173, Accuracy: 0.95


train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇████
train/example_ct,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▅▅▆▆▆▆▆▇▇▇▇▇████
train/train_loss,█▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/val_accuracy,▁▄▆▇▇█████
val/val_loss,█▅▄▃▂▁▂▁▁▁
train/epoch,10
train/example_ct,120000
train/train_loss,0.02928
val/val_accuracy,0.954
val/val_loss,0.14817


테이블이 생성된걸 확인할 수 있습니다.
- filter를 사용해 정답이 틀린 것만 확인해 봅시다
- `col1 != col2` (pred != target)
- 다양한 operation를 이용해 filter를 사용할 수 있습니다.

![](https://i.imgur.com/NGSNlwg.png)

### 4.3. Examples

#### [Compare predicted scores for specific images](https://wandb.ai/stacey/table-quickstart/reports/CNN-2-Progress-over-Training-Time--Vmlldzo3NDY5ODU#compare-predictions-after-1-vs-5-epochs)

#### [Focus on top errors over time](https://wandb.ai/stacey/table-quickstart/reports/CNN-2-Progress-over-Training-Time--Vmlldzo3NDY5ODU#top-errors-over-time)

#### [Compare model performance and find patterns](https://wandb.ai/stacey/table-quickstart/reports/CNN-2-Progress-over-Training-Time--Vmlldzo3NDY5ODU#false-positives-grouped-by-guess)


## 5. Hyperparameter Sweeps

[video tutorial](http://wandb.me/sweeps-video)

![](https://i.imgur.com/WVKkMWw.png)

앞 섹션에서는 dropout 하나만 무작위로 변경하여 5회 실험했습니다. 실제 환경에서는 여러 hyperparameter의 조합을 탐색해야 하는 경우가 많습니다.


**Sweep**은 hyperparameter 조합을 자동으로 탐색하는 W&B 기능입니다. 세 단계로 구성됩니다.

1. **Sweep 정의**: 탐색할 parameter, 탐색 전략, 최적화 metric을 dict 또는 YAML로 정의
2. **Sweep 초기화**: `wandb.sweep(config)` 호출하여 `sweep_id` 획득
3. **Sweep agent 실행**: `wandb.agent(sweep_id, function=train)` 호출

### Sweep 아키텍처

- **Sweep Controller** (W&B 서버 측): 다음에 시도할 config 결정
- **Agent** (로컬 머신 측): Controller에 config 요청 후 받은 config로 학습 실행, 결과 보고

<img src="https://i.imgur.com/zlbw3vQ.png" width=350 />

### 탐색 전략 (method)

- `grid`: 모든 조합 시도 (계산 비용 증가)
- `random`: 무작위 샘플링 (가장 일반적으로 사용)
- `bayes`: Bayesian optimization, 이전 결과를 바탕으로 다음 config 결정 (적은 수의 연속형 parameter에 효과적)

**Resources**
- [Sweeps docs →](https://docs.wandb.ai/sweeps)
- [Launching from the command line →](https://www.wandb.com/articles/hyperparameter-tuning-as-easy-as-1-2-3)

### 5.1 Sweep 정의

In [ ]:
# 탐색 전략 선택
sweep_config = {
    'method': 'random'  # 'random', 'grid', 'bayes' 중 선택
}

# 최적화 대상 metric (bayes 사용 시 필수)
metric = {
    'name': 'loss',
    'goal': 'minimize'  # 'minimize' 또는 'maximize'
}
sweep_config['metric'] = metric

### 5.2 탐색 Parameter 정의

Parameter는 두 가지 방식으로 정의할 수 있습니다.

- `values: [...]`: 명시적 후보 리스트
- `distribution: 'uniform'` 등: 분포에서 샘플링 ([전체 분포 종류](https://docs.wandb.com/sweeps/configuration#distributions))

In [ ]:
parameters_dict = {
    # 카테고리형: values 리스트에서 선택
    'optimizer':     {'values': ['adam', 'sgd']},
    'fc_layer_size': {'values': [128, 256, 512]},
    'dropout':       {'values': [0.3, 0.4, 0.5]},
}
sweep_config['parameters'] = parameters_dict

# 고정값 (탐색하지 않고 모든 run에 동일하게 적용)
parameters_dict.update({
    'epochs': {'value': 1}
})

# 연속형: 분포에서 샘플링
parameters_dict.update({
    'learning_rate': {
    # 값을 로그 스케일로 탐색하여 작은 값(예: 1e-4, 1e-3)을 더 촘촘히 테스트합니다.
    'distribution': 'log_uniform_values',
    'max': 0.01,   # 최대값을 기존 0.1에서 0.01로 대폭 낮춤
    'min': 0.0001,  # 최소값 설정 (1e-4)
},

# parameters_dict.update({
#     'learning_rate': {
#         'distribution': 'uniform',
#         'min': 0,
#         'max': 0.1,
#     },
    'batch_size': {
        # 32~256 범위의 정수, log-uniform 분포에서 8의 배수로 양자화
        'distribution': 'q_log_uniform_values',
        'q': 8,
        'min': 32,
        'max': 256,
    },
})

import pprint
pprint.pprint(sweep_config)

{'method': 'random',
 'metric': {'goal': 'minimize', 'name': 'loss'},
 'parameters': {'batch_size': {'distribution': 'q_log_uniform_values',
                               'max': 256,
                               'min': 32,
                               'q': 8},
                'dropout': {'values': [0.3, 0.4, 0.5]},
                'epochs': {'value': 1},
                'fc_layer_size': {'values': [128, 256, 512]},
                'learning_rate': {'distribution': 'log_uniform_values',
                                  'max': 0.01,
                                  'min': 0.0001},
                'optimizer': {'values': ['adam', 'sgd']}}}


### 5.3 Sweep 초기화

`wandb.sweep()`은 Sweep Controller를 W&B 서버에 등록하고 `sweep_id`를 반환합니다.

In [ ]:
sweep_id = wandb.sweep(sweep_config, project="pytorch-sweeps-demo")

Create sweep with ID: 2d32vfqf
Sweep URL: https://wandb.ai/jaehyeon6874-2/pytorch-sweeps-demo/sweeps/2d32vfqf


### 5.4 학습 함수 정의

Sweep에서는 `train(config=None)` 형태의 함수를 정의하고, 그 내부에서 `wandb.init(config=config)`를 호출합니다. Agent가 이 함수를 호출할 때 Controller가 제공한 config가 `wandb.config`로 자동 주입됩니다.

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train(config=None):
    # with 블록: run이 자동으로 finish됨 (sweep에서 권장되는 패턴)
    with wandb.init(config=config):
        # agent가 전달한 config가 여기로 들어옴
        config = wandb.config

        loader = build_dataset(config.batch_size)
        network = build_network(config.fc_layer_size, config.dropout)
        optimizer = build_optimizer(network, config.optimizer, config.learning_rate)

        for epoch in range(config.epochs):
            avg_loss = train_epoch(network, loader, optimizer)
            wandb.log({"loss": avg_loss, "epoch": epoch})


def build_dataset(batch_size):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    dataset = datasets.MNIST(".", train=True, download=True, transform=transform)
    sub_dataset = torch.utils.data.Subset(dataset, indices=range(0, len(dataset), 5))
    return torch.utils.data.DataLoader(sub_dataset, batch_size=batch_size)


def build_network(fc_layer_size, dropout):
    network = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, fc_layer_size), nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(fc_layer_size, 10),
        nn.LogSoftmax(dim=1),
    )
    return network.to(device)


def build_optimizer(network, optimizer, learning_rate):
    if optimizer == "sgd":
        return optim.SGD(network.parameters(), lr=learning_rate, momentum=0.9)
    elif optimizer == "adam":
        return optim.Adam(network.parameters(), lr=learning_rate)


def train_epoch(network, loader, optimizer):
    cumu_loss = 0
    for _, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = F.nll_loss(network(data), target)
        cumu_loss += loss.item()
        loss.backward()
        optimizer.step()
        wandb.log({"batch loss": loss.item()})
    return cumu_loss / len(loader)

### 5.5 Sweep Agent 실행

`count=5`는 5개의 config를 요청하여 실행함을 의미합니다. 생략 시 무한히 실행되므로 수동 중지가 필요합니다.

동일한 `sweep_id`로 여러 머신 또는 Colab 세션에서 agent를 실행하면 병렬 실행이 가능합니다.

In [ ]:
wandb.agent(sweep_id, train, count=5)

wandb: Agent Starting Run: ysdekme9 with config:
wandb: 	batch_size: 248
wandb: 	dropout: 0.4
wandb: 	epochs: 1
wandb: 	fc_layer_size: 128
wandb: 	learning_rate: 0.0006640762174585394
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


batch loss,█████▇█▇▇▇▇▆▆▆▆▆▆▆▆▅▅▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▁
epoch,▁
loss,▁
batch loss,1.84764
epoch,0
loss,2.13891


wandb: Agent Starting Run: gjs8hm8x with config:
wandb: 	batch_size: 88
wandb: 	dropout: 0.5
wandb: 	epochs: 1
wandb: 	fc_layer_size: 256
wandb: 	learning_rate: 0.0016329169365399308
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


batch loss,██▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▄▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁
epoch,▁
loss,▁
batch loss,0.68002
epoch,0
loss,1.3032


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: nm6fownd with config:
wandb: 	batch_size: 80
wandb: 	dropout: 0.5
wandb: 	epochs: 1
wandb: 	fc_layer_size: 512
wandb: 	learning_rate: 0.0014611173894176213
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


batch loss,██▇▇▇▆▆▆▅▆▆▅▅▅▅▅▄▅▄▃▃▃▃▄▄▂▂▂▂▂▁▁▂▂▁▁▂▂▁▁
epoch,▁
loss,▁
batch loss,0.54411
epoch,0
loss,1.20045


wandb: Agent Starting Run: 7pcmeekg with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.5
wandb: 	epochs: 1
wandb: 	fc_layer_size: 256
wandb: 	learning_rate: 0.0005169507392274838
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


batch loss,██▇▇█▇▇▇▇▆▆▆▇▆▆▅▆▅▅▄▄▄▄▄▄▂▂▄▃▂▂▂▂▂▂▂▂▁▂▁
epoch,▁
loss,▁
batch loss,1.13636
epoch,0
loss,1.72293


wandb: Agent Starting Run: 23xtdstn with config:
wandb: 	batch_size: 56
wandb: 	dropout: 0.5
wandb: 	epochs: 1
wandb: 	fc_layer_size: 512
wandb: 	learning_rate: 0.0016235559425470038
wandb: 	optimizer: adam
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


batch loss,█▄▅▃▂▂▂▃▂▃▃▃▃▃▁▂▃▂▃▃▂▂▂▂▃▁▂▂▂▂▁▂▂▂▃▂▁▂▁▁
epoch,▁
loss,▁
batch loss,0.05731
epoch,0
loss,0.45946


### 5.6 Sweep 결과 시각화

Sweep 페이지에서 자동으로 생성되는 두 가지 주요 시각화는 다음과 같습니다.

#### Parallel Coordinates Plot

각 hyperparameter 값을 축으로 하여 모델 metric에 매핑합니다. 최고 성능을 보인 hyperparameter 조합을 찾는 데 유용합니다.

![](https://assets.website-files.com/5ac6b7f2924c652fd013a891/5e190366778ad831455f9af2_s_194708415DEC35F74A7691FF6810D3B14703D1EFE1672ED29000BA98171242A5_1578695138341_image.png)

#### Hyperparameter Importance Plot

어떤 hyperparameter가 metric에 가장 큰 영향을 미쳤는지 보여줍니다. Random forest 기반의 feature importance와 선형 모델 기반의 correlation을 함께 확인할 수 있습니다.

![](https://assets.website-files.com/5ac6b7f2924c652fd013a891/5e190367778ad820b35f9af5_s_194708415DEC35F74A7691FF6810D3B14703D1EFE1672ED29000BA98171242A5_1578695757573_image.png)

이러한 시각화는 가장 중요한 parameter와 효과적인 value 범위를 파악함으로써 후속 hyperparameter 탐색에 드는 시간과 자원을 절약하는 데 도움을 줍니다.

## 6. 추가 학습 주제

- **Bayesian sweep**: `method`를 `'bayes'`로 변경하여 동일한 코드를 재실행합니다. Random보다 효율적으로 좋은 config를 탐색합니다.
- **`wandb.Artifact`**: 데이터셋 및 모델의 버전 관리 기능입니다.
- **W&B Reports**: 동료와 공유 가능한 인터랙티브 분석 문서입니다.
- **CLI 기반 Sweep**: `wandb sweep config.yaml` 후 `wandb agent <sweep_id>`로 여러 서버에서 병렬 분산 실행이 가능합니다.

### 참고 문서

- [Sweeps Documentation](https://docs.wandb.ai/sweeps)
- [Artifacts Documentation](https://docs.wandb.ai/guides/artifacts)
- [Reports Documentation](https://docs.wandb.ai/guides/reports)